In [ ]:
# @title Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
# https://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

# Psyre Graph Causality Analysis, Clock Calculus, and Stage-1 Visualization

This notebook demonstrates Stage-1 static analysis, clock pretty-printing, and interactive schematic visualization on Psyre dataflow `Graph`s (`Scheduler.build_graph`) prior to Stage-2 execution (`Scheduler.run_graph`):

- **Causality analysis** (`analysis.check_causality()`, `analysis.topological_sort()`, `CausalityError`): verifies that every instantaneous dependency cycle is broken by a unit delay (`fby` / `pre` / `last`), and extracts a shortest elementary cycle when violated.
- **Synchronous clock calculus & pretty-printing** (`Clock`, `ClockAnalysis`, `clock.check_clocks()`, `clocks.stream_clock()`, `ClockError`): infers Lustre / Lucid Synchrone hierarchical subclocks across `when` / `whennot`, `merge`, `buffer`, `reset`, `match`, and `automaton`, pretty-printing clocks via `Clock.__str__()` using `.` for the base clock, quoted variables (`'a`, `'b`, ...) for polymorphic constants, and synchronous `on` / `on not` / `on (...)` syntax for subclocks.
- **Interactive Stage-1 schematic visualization** (`visualize.visualize_graph()`): renders lowered `SYNC` / `KAHN` primitive graphs with selector/multiplexor trapezoids for `when`/`whennot`/`merge` and square boxes with top-centered inverted hats (`∨`) for `fby`/`pre`, plus collapsible hierarchical automaton subgraphs.

In [ ]:
# @title Psyre imports and graph inspection helpers

from collections.abc import Callable, Mapping
import logging
from typing import Any
# !pip install -q psyre

# pylint: disable=g-importing-member,g-multiple-import
from psyre.analysis import (
    CausalityError,
    check_causality,
    topological_sort,
)
from psyre.automaton import (
    automaton,
    state,
    unless,
    until,
)
from psyre.clock import (
    BASE_CLOCK,
    Clock,
    ClockError,
    check_clocks,
)
from psyre.graph import Graph
from psyre.match import (
    case,
    default,
    last,
    match,
)
from psyre.psyre import (
    Scheduler,
    Semantics,
    Stream,
    buffer,
    counter,
    fby,
    ifthenelse,
    logger,
    merge,
    pre,
    psyre_comb,
    reset,
    scan,
    when,
    whennot,
)
from psyre.visualize import visualize_graph

# pylint: enable=g-importing-member,g-multiple-import

logger.setLevel(logging.WARNING)
print('BASE_CLOCK:', BASE_CLOCK)
print('Polymorphic clocks:', Clock.polymorphic(0), Clock.polymorphic(1))


def inspect_graph(title: str, g: Graph, tracked: Mapping[str, Stream]) -> None:
  """Validates causality and clocks on g and prints order and stream clocks."""
  check_causality(g)
  order = topological_sort(g)
  clocks = check_clocks(g)
  sem_name = g.semantics.name if g.semantics is not None else 'DEFAULT'
  print(f'=== {title} [{sem_name}] ({len(order)} nodes) ===')
  print('  Topological order:', ' -> '.join(str(node) for node in order))
  for label, s in tracked.items():
    clk = clocks.stream_clock(s)
    print(f'  clock({label}) = {clk}')


def expect_error(
    title: str, exc_type: type[Exception], fn: Callable[[], Any]
) -> None:
  """Asserts that fn() raises exc_type and prints the diagnostic."""
  try:
    fn()
  except exc_type as err:
    print(f'[{title}] {exc_type.__name__}: {err}')
    return
  raise AssertionError(f'Expected {exc_type.__name__} for {title!r}.')

## 1. Programs Without Automata (`when`, `merge`, `match`, `fby`, `buffer`)

Programs without stateful `automaton()` blocks run under both `Semantics.KAHN` and `Semantics.SYNC`. Whenever `check_causality()` and `check_clocks()` succeed, `KAHN` execution coincides with the `Absent`-elision of `SYNC` execution.

In [ ]:
# @title 1.1 Positive causality and clock calculus without automata


@psyre_comb
def add_two(v: Any) -> Any:
  return v + 2


def pair_box(u: Any, v: Any) -> tuple[Stream, Stream]:
  """Two-port box flattened into independent primitive nodes (u -> a -> b)."""
  return u + 1, add_two(v)


tracked_a: dict[str, Stream] = {}


def when_merge_feedback() -> tuple[Stream, Stream, Stream, Stream]:
  x = Stream([10, 20, 30, 40], name='x')
  k_scale = Stream(2, name='k_scale')
  k_bias = Stream(0, name='k_bias')
  odd_p = ((x // 10) & 1) == 1
  odd_p.name = 'odd_p'
  odd = when(odd_p, x + k_bias, name='odd')
  even = whennot(odd_p, x * k_scale, name='even')
  rejoined = merge(odd_p, odd, even, name='rejoined')
  acc = fby(0, lambda: nxt, name='acc')  # pylint: disable=undefined-variable
  nxt = acc + rejoined
  a, b = pair_box(x, lambda: a)  # pylint: disable=undefined-variable
  tracked_a.update(
      k_scale=k_scale,
      k_bias=k_bias,
      x=x,
      odd=odd,
      even=even,
      rejoined=rejoined,
      nxt=nxt,
      b=b,
  )
  return odd, rejoined, nxt, b


for sem in (Semantics.KAHN, Semantics.SYNC):
  g_a = Scheduler.build_graph(when_merge_feedback, semantics=sem)
  inspect_graph(
      'when/merge + fby feedback + flattened pair_box', g_a, tracked_a
  )
  print('  Traces:', Scheduler.run_graph(g_a, horizon=4), '\n')


tracked_b: dict[str, Stream] = {}


def match_last_and_buffer() -> tuple[Stream, Stream]:
  cmd = Stream(['ADD', 'HOLD', 'SUB', 'DEC'], name='cmd')
  x = Stream([10, 20, 30, 40], name='x')
  with match(cmd) as m:
    prev = last(lambda: m.val, init=0, name='prev')
    with case('ADD'):
      add_arm = prev + x
      m.val = add_arm
    with case('SUB', 'DEC'):
      sub_arm = prev - 5
      m.val = sub_arm
    with default():
      def_arm = prev + 0
      m.val = def_arm

  ip = Stream([True, False, True, False], name='ip')
  op = Stream([False, True, False, True], name='op')
  buf = buffer(when(ip, x), ip, op, size=2, name='buf')
  with match(op) as buf_m:
    with case(True):
      consumed = buf + 1
      buf_m.out = consumed
    with case(False):
      buf_m.out = 0

  tracked_b.update(
      add_arm=add_arm,
      sub_arm=sub_arm,
      def_arm=def_arm,
      match_val=m.val,
      buf=buf,
      consumed=consumed,
  )
  return m.val, buf_m.out


for sem in (Semantics.KAHN, Semantics.SYNC):
  g_b = Scheduler.build_graph(match_last_and_buffer, semantics=sem)
  inspect_graph('stateless match/last + rate-bridging buffer', g_b, tracked_b)
  print('  Traces:', Scheduler.run_graph(g_b, horizon=4), '\n')

In [ ]:
# @title 1.2 Negative causality and clock calculus without automata


def combinational_two_node_cycle() -> Stream:
  a = ifthenelse(True, lambda: b, 0)  # pylint: disable=undefined-variable
  b = a + 1
  return a


def fby_initial_value_cycle() -> Stream:
  # Only fby's second argument is unit-delayed; its initial value is not.
  x = fby(lambda: x + 1, 0)  # pylint: disable=undefined-variable
  return x


def fby_reset_control_cycle() -> Stream:
  with reset(lambda: y > 0):  # pylint: disable=undefined-variable
    y = fby(0, 1, name='y')
  return y


def match_selector_cycle() -> Stream:
  with match(lambda: m.out) as m:
    with case(True):
      m.out = False
    with case(False):
      m.out = True
  return m.out


print('--- Negative causality without automata ---')
for fn in (
    combinational_two_node_cycle,
    fby_initial_value_cycle,
    fby_reset_control_cycle,
    match_selector_cycle,
):
  expect_error(
      fn.__name__,
      CausalityError,
      lambda prog=fn: check_causality(
          Scheduler.build_graph(prog, semantics=Semantics.KAHN)
      ),
  )


def disjoint_subclocks() -> Stream:
  c = Stream([True, False, True, False], name='c')
  x = Stream([1, 2, 3, 4], name='x')
  return when(c, x) + whennot(c, x)


def unguarded_merge_arm() -> Stream:
  c = Stream([True, False, True, False], name='c')
  x = Stream([1, 2, 3, 4], name='x')
  return merge(c, counter(), whennot(c, x))


def stateful_without_match() -> Stream:
  c = Stream([True, False, True, False], name='c')
  x = Stream([1, 2, 3, 4], name='x')
  return scan(when(c, x))


def redundant_when_inside_match() -> Stream:
  c = Stream([True, False, True, False], name='c')
  x = Stream([1, 2, 3, 4], name='x')
  y = Stream([10, 20, 30, 40], name='y')
  with match(c) as m:
    with case(True):
      m.out = when(c, x) + y
    with case(False):
      m.out = 0
  return m.out


def subclock_reset_gate() -> Stream:
  c = Stream([True, False, True, False], name='c')
  r = Stream([False, True, False, False], name='r')
  with reset(when(c, r)):
    return counter()


def cross_activity_reset() -> Stream:
  c = Stream([True, False, True, False], name='c')
  r = Stream([False, True, False, False], name='r')
  with reset(r):
    with match(c) as m:
      with case(True):
        m.out = counter()
      with case(False):
        m.out = 0
  return m.out


print('\n--- Negative clock calculus without automata ---')
for fn in (
    disjoint_subclocks,
    unguarded_merge_arm,
    stateful_without_match,
    redundant_when_inside_match,
    subclock_reset_gate,
):
  expect_error(
      fn.__name__,
      ClockError,
      lambda prog=fn: check_clocks(
          Scheduler.build_graph(prog, semantics=Semantics.SYNC)
      ),
  )
expect_error(
    'cross_activity_reset [KAHN]',
    ClockError,
    lambda: check_clocks(
        Scheduler.build_graph(cross_activity_reset, semantics=Semantics.KAHN)
    ),
)

## 2. Programs With Automata (`automaton`, `state`, `until`, `unless`, `last`)

Stateful `automaton()` blocks execute under `Semantics.SYNC` and lower into `fby`-delayed state/reset registers (`pns`, `pnr`), prioritized strong preemptions (`unless`), per-state activation and `reset` scopes, weak post-step transitions (`until`), and output `merge`s. `check_causality()` and `check_clocks()` analyze the full lowered network and pretty-print hierarchical state subclocks.

In [ ]:
# @title 2.1 Positive causality and clock calculus with automata

tracked_fsm: dict[str, Stream] = {}


def well_clocked_automaton() -> Stream:
  inp = Stream([1, 2, 3, 4, 5], name='inp')
  k_bonus = Stream(0, name='k_bonus')
  tag_sel = Stream([True, False, True, False, True], name='tag_sel')
  with automaton() as fsm:
    prev = last(lambda: fsm.out, init=0, name='prev')
    with state('RUN', unless(prev >= 6, 'DONE')):
      cnt = counter(start=1, name='run_cnt')
      with match(tag_sel) as bonus:
        with case(True):
          bonus_true = inp + k_bonus
          bonus.v = bonus_true
        with case(False):
          bonus.v = 0
      run_val = cnt + bonus.v
      # Weak transitions feed fby-delayed `pns`, so reading `fsm.out` is causal:
      until(lambda: fsm.out > 3, 'HOLD')
      fsm.out = run_val
    with state('HOLD'):
      hold_val = prev + 1
      until(True, 'RUN', reset=False)
      fsm.out = hold_val
    with state('DONE'):
      fsm.out = 99

  tracked_fsm.update(
      k_bonus=k_bonus,
      prev=prev,
      run_val=run_val,
      bonus_true=bonus_true,
      hold_val=hold_val,
      fsm_out=fsm.out,
  )
  return fsm.out


g_fsm = Scheduler.build_graph(well_clocked_automaton, semantics=Semantics.SYNC)
inspect_graph(
    '3-state automaton (unless + until + last + stateless match)',
    g_fsm,
    tracked_fsm,
)
print('  Trace:', Scheduler.run_graph(g_fsm, horizon=5))

In [ ]:
# @title 2.2 Negative causality and clock calculus with automata


def strong_unless_output_cycle() -> Stream:
  # Strong `unless` preempts the active state before body evaluation in the
  # same tick, so reading current-tick `fsm.out` forms an instantaneous cycle.
  with automaton() as fsm:
    with state(False, unless(lambda: fsm.out, True)):
      fsm.out = True
    with state(True):
      fsm.out = False
  return fsm.out


def intra_state_combinational_cycle() -> Stream:
  with automaton() as fsm:
    with state('A'):
      v = ifthenelse(True, lambda: fsm.out, 0)
      fsm.out = v + 1
  return fsm.out


print('--- Negative causality with automata ---')
for fn in (strong_unless_output_cycle, intra_state_combinational_cycle):
  expect_error(
      fn.__name__,
      CausalityError,
      lambda prog=fn: check_causality(
          Scheduler.build_graph(prog, semantics=Semantics.SYNC)
      ),
  )


def subclock_state_output() -> Stream:
  sub = Stream([True, False, True, False], name='sub')
  x = Stream([10, 20, 30, 40], name='x')
  with automaton() as fsm:
    with state('A'):
      until(False, 'B')
      fsm.out = when(sub, x)
    with state('B'):
      fsm.out = 0
  return fsm.out


def subclock_until_guard() -> Stream:
  sub = Stream([True, False, True, False], name='sub')
  with automaton() as fsm:
    with state('A'):
      until(when(sub, True), 'B')
      fsm.out = 1
    with state('B'):
      fsm.out = 2
  return fsm.out


def subclock_unless_guard() -> Stream:
  sub = Stream([True, False, True, False], name='sub')
  with automaton() as fsm:
    with state('A', unless(when(sub, True), 'B')):
      fsm.out = 1
    with state('B'):
      fsm.out = 2
  return fsm.out


def subclock_state_reset() -> Stream:
  sub = Stream([True, False, True, False], name='sub')
  with automaton() as fsm:
    with state('A'):
      with reset(when(sub, True)):
        fsm.out = counter()
    with state('B'):
      fsm.out = 0
  return fsm.out


def hierarchical_modular_reset() -> Stream:
  outer_flip = Stream([False, True, True, False], name='outer_flip')
  inner_flip = Stream([True, False, False, False], name='inner_flip')
  with automaton() as outer:
    with state('ON'):
      with automaton() as inner:
        with state('X'):
          until(inner_flip, 'Y', reset=False)
          inner.v = counter(start=1)
        with state('Y'):
          until(inner_flip, 'X', reset=False)
          inner.v = counter(start=10)
      until(outer_flip, 'OFF')
      outer.out = inner.v
    with state('OFF'):
      until(outer_flip, 'ON')
      outer.out = -1
  return outer.out


print('\n--- Negative clock calculus with automata ---')
for fn in (
    subclock_state_output,
    subclock_until_guard,
    subclock_unless_guard,
    subclock_state_reset,
):
  expect_error(
      fn.__name__,
      ClockError,
      lambda prog=fn: check_clocks(
          Scheduler.build_graph(prog, semantics=Semantics.SYNC)
      ),
  )

hier_g = Scheduler.build_graph(
    hierarchical_modular_reset, semantics=Semantics.SYNC
)
check_causality(hier_g)
hier_clocks = check_clocks(hier_g)
print(
    '\nSYNC hierarchical modular-reset clock & runtime trace:',
    hier_clocks.stream_clock(hier_g.output),
    Scheduler.run_graph(hier_g, horizon=4),
)

## 3. Interactive Stage-1 Schematic Visualization

`visualize.visualize_graph(g)` renders lowered Stage-1 `SYNC` and `KAHN` graphs as an interactive pan/zoom schematic:
- **`when` / `whennot`**: narrow-to-wide selector trapezoids (with an inversion bubble on `whennot`).
- **`merge`**: wide-to-narrow multiplexor trapezoids with per-arm tags.
- **`ifthenelse`**: flow-graph diamond (rhombus) with `T` and `F` arm labels.
- **`fby` / `pre` / `post`**: `1:1` square registers with an inverted hat (`∨`) centered along the top edge for `fby`/`pre` or a hat (`∧`) centered along the bottom edge for `post`.
- **`morph`**: stateful double-bordered box (`morph`, or `morph` with subscript $n$ when $n > 1$).
- **`prefix_sum` / `scan`**: rectangle box with a capital Sigma (`Σ`) inside.
- **`delay` / `buffer`**: slotted Kahn Process Network FIFO buffer symbol.
- **Periodic / static streams & `once_stream(k)` ($\delta_k$)**: boxless plain text next to a short arrow pointing directly into each consuming node (re-instantiated per consumer).
- **Interactive controls**: left-pane zoom slider or pinch-to-zoom, drag to pan, `+`/`−`/`Fit`/`1:1` buttons, click automaton summary boxes to expand/collapse subgraphs, and hover/click any node or wire to inspect its inferred clock and neighborhood.

In [ ]:
# @title 3.1 Interactive schematic: when/whennot, merge, fby, and pre (KAHN)


def selector_mux_delay_demo() -> tuple[Stream, Stream]:
  mask = Stream([True, False, True, False], name='mask')
  vals = Stream([10, 20, 30, 40], name='vals')
  sampled = when(mask, vals, name='sampled')
  complement = whennot(mask, add_two(vals), name='complement')
  merged = merge(mask, sampled, complement, name='merged')
  prev_merged = pre(merged, name='pre')
  acc = fby(0, lambda: nxt, name='acc')  # pylint: disable=undefined-variable
  nxt = acc + merged
  return prev_merged, nxt


g_schematic = Scheduler.build_graph(
    selector_mux_delay_demo, semantics=Semantics.KAHN
)
visualize_graph(
    g_schematic,
    title='Stage-1 KAHN Primitives: when/whennot, merge, fby, and pre',
    height_px=460,
)

In [ ]:
# @title 3.2 Lowered 3-state SYNC automaton schematic

visualize_graph(
    g_fsm,
    title='Lowered Stage-1 SYNC Automaton (3 states + nested match)',
    height_px=620,
)